# VinAI Translate baseline trên Google Colab

Notebook chạy cùng code/config với máy local: kiểm tra dữ liệu và tokenizer, smoke test, sau đó zero-shot hoặc fine-tune LoRA/full. Trước khi chạy, chọn **Runtime → Change runtime type → GPU**. Public test không được dùng để huấn luyện hoặc chọn checkpoint.

In [ ]:
import subprocess, sys
subprocess.run(["nvidia-smi"], check=False)
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU runtime trước khi tiếp tục.")
print("GPU:", torch.cuda.get_device_name(0))

## 1. Gắn Google Drive

Kết quả full run, checkpoint và model tốt nhất được ghi vào `MyDrive/NLP_runs`. Nếu Drive đã được gắn, Colab có thể hỏi xác nhận dùng lại mount hiện tại.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Upload và giải nén mã nguồn

Upload đúng một file `NLP_colab.zip` được tạo bởi `scripts/package_colab.ps1`. Cell hỗ trợ cả ZIP Windows dùng dấu `\` và ZIP có một thư mục bao ngoài. Thành công khi cuối cell in ra đường dẫn `Project:`.

In [ ]:
from pathlib import Path, PurePosixPath
import os, shutil, subprocess, tempfile, zipfile
from google.colab import files

# Private repo: nén thư mục dự án thành một file .zip rồi chọn file đó ở đây.
# ZIP phải chứa requirements-training.txt, thư mục src/ và configs/.
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Hãy upload đúng một file ZIP chứa mã nguồn dự án.")

zip_name = zip_names[0]
zip_path = Path("/content") / Path(zip_name).name
zip_path.write_bytes(uploaded[zip_name])
extract_dir = Path(tempfile.mkdtemp(prefix="nlp_upload_", dir="/content"))
extract_root = extract_dir.resolve()

with zipfile.ZipFile(zip_path) as archive:
    for member in archive.infolist():
        # Compress-Archive trên Windows có thể lưu dấu \\ trong tên entry.
        # Colab/Linux không xem \\ là dấu phân cách, nên phải chuẩn hóa sang /.
        normalized = member.filename.replace("\\", "/")
        relative = PurePosixPath(normalized)
        if (
            relative.is_absolute()
            or ".." in relative.parts
            or (relative.parts and ":" in relative.parts[0])
        ):
            raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
        if not relative.parts:
            continue
        destination = extract_dir.joinpath(*relative.parts)
        if not destination.resolve().is_relative_to(extract_root):
            raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
        if member.is_dir() or normalized.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)

def is_project_root(path):
    return (
        (path / "requirements-training.txt").is_file()
        and (path / "src" / "vinai_baseline" / "cli.py").is_file()
        and (path / "configs" / "vinai").is_dir()
    )

# ZIP tạo bằng lệnh đóng gói được khuyến nghị có project ngay tại thư mục gốc.
# Vẫn hỗ trợ ZIP có thêm một thư mục bao ngoài như NLP-main/.
candidates = [extract_dir] if is_project_root(extract_dir) else []
candidates.extend(
    path.parent
    for path in extract_dir.rglob("requirements-training.txt")
    if path.parent != extract_dir and is_project_root(path.parent)
)
candidates = sorted(set(candidates), key=lambda path: (len(path.parts), str(path)))
if not candidates:
    extracted = sorted(str(path.relative_to(extract_dir)) for path in extract_dir.iterdir())
    raise FileNotFoundError(
        "Không tìm thấy gốc dự án trong ZIP. Các mục ở cấp đầu: " + repr(extracted)
    )
if len(candidates) > 1:
    print("Tìm thấy nhiều thư mục dự án; dùng thư mục ngoài cùng:", candidates[0])
PROJECT_DIR = candidates[0]
os.chdir(PROJECT_DIR)
print("Đã giải nén ZIP:", zip_path.name)
print("Project:", Path.cwd())

## 3. Cài thư viện

Cell này cài đúng phiên bản thư viện của baseline và tự gỡ `torchao` cũ có sẵn trong Colab nếu phiên bản đó xung đột với PEFT. Baseline LoRA không dùng TorchAO/quantization. Nếu Colab yêu cầu restart runtime, hãy restart rồi chạy lại notebook từ đầu vì các biến `PROJECT_DIR`, `CONFIG` và `OUTPUT_DIR` sẽ bị mất.

In [ ]:
requirement_file = (PROJECT_DIR / "requirements-training-colab.txt"
                    if (PROJECT_DIR / "requirements-training-colab.txt").exists()
                    else PROJECT_DIR / "requirements-training.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirement_file)], check=True)

# Colab có thể cài sẵn torchao 0.10.0, trong khi PEFT 0.21 chỉ chấp nhận >0.16.
# Dự án không dùng TorchAO nên gỡ bản xung đột là phương án ít rủi ro nhất.
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
try:
    torchao_version = version("torchao")
except PackageNotFoundError:
    torchao_version = None
if torchao_version and Version(torchao_version) <= Version("0.16.0"):
    print(f"Gỡ torchao {torchao_version} vì xung đột với PEFT 0.21...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
elif torchao_version:
    print("torchao tương thích:", torchao_version)
else:
    print("torchao không được cài (phù hợp vì baseline không sử dụng).")
print("Đã cài dependency từ:", requirement_file.name)
print("Nếu pip yêu cầu restart runtime, hãy restart rồi chạy lại từ cell GPU.")

## 4. Hugging Face token (không bắt buộc)

Model VinAI là public. `HF_TOKEN` chỉ giúp tải ổn định hơn; thiếu token không phải lỗi training. Có thể thêm token trong **Colab → Secrets** với tên `HF_TOKEN`.

In [ ]:
# Model public nên HF_TOKEN không bắt buộc; token giúp hạn mức tải ổn định hơn.
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("Đã nạp HF_TOKEN từ Colab Secrets.")
except Exception:
    print("Không có HF_TOKEN; tiếp tục tải model public.")

## 5. Chọn thí nghiệm

- `DIRECTION`: `en2vi` hoặc `vi2en`.
- `MODE`: `zero_shot`, `lora` hoặc `full`.
- Giữ `RUN_FULL=False` để chạy smoke test trước. Chỉ đổi thành `True` sau khi smoke thành công.
- Khi resume, điền đúng tên run cũ vào `RUN_ID_OVERRIDE`, đặt `RESUME_FROM_CHECKPOINT='last'` và `RUN_FULL=True`.

Seed split và training được đọc từ config và phải là **67**. Mỗi thành viên dùng `MEMBER_TAG` riêng để không ghi đè kết quả.

In [ ]:
# Các biến điều khiển run. MODE: zero_shot | lora | full
DIRECTION = "en2vi"
MODE = "lora"
CONFIG_OVERRIDE = ""  # ví dụ configs/generated_v3/en2vi_lora.json
RUN_FULL = False  # đổi True sau khi smoke test thành công
MEMBER_TAG = "thuan"  # mỗi thành viên dùng một tên ASCII riêng
EXPERIMENT_TAG = "baseline_v2_t4safe16"
RUN_ID_OVERRIDE = ""  # điền run ID cũ khi cần resume đúng output directory
RESUME_FROM_CHECKPOINT = ""  # để trống khi chạy mới; dùng 'last' khi Colab bị ngắt

import json, re
from datetime import datetime, timezone
if DIRECTION not in {"en2vi", "vi2en"}:
    raise ValueError("DIRECTION phải là en2vi hoặc vi2en")
if MODE not in {"zero_shot", "lora", "full"}:
    raise ValueError("MODE phải là zero_shot, lora hoặc full")
for label, value in {"MEMBER_TAG": MEMBER_TAG, "EXPERIMENT_TAG": EXPERIMENT_TAG}.items():
    if not re.fullmatch(r"[a-z0-9_-]+", value):
        raise ValueError(f"{label} chỉ dùng chữ thường ASCII, số, _ hoặc -")
if RUN_ID_OVERRIDE and not re.fullmatch(r"[a-z0-9_-]+", RUN_ID_OVERRIDE):
    raise ValueError("RUN_ID_OVERRIDE phải là đúng tên thư mục run, không chứa dấu phân cách thư mục")
CONFIG = (PROJECT_DIR / CONFIG_OVERRIDE if CONFIG_OVERRIDE else PROJECT_DIR / "configs" / "vinai" / f"{DIRECTION}_{MODE}.json")
config_preview = json.loads(CONFIG.read_text(encoding="utf-8"))
expected_training_mode = "none" if MODE == "zero_shot" else MODE
if config_preview["direction"] != DIRECTION or config_preview["training"]["mode"] != expected_training_mode:
    raise ValueError("DIRECTION/MODE không khớp với CONFIG_OVERRIDE")
DATASET_ID = config_preview["data"]["dataset_id"]
TRAIN_SEED = config_preview["seed"]
DATA_DIR = PROJECT_DIR / Path(config_preview["data"]["manifest_file"]).parent
if RESUME_FROM_CHECKPOINT and not RUN_ID_OVERRIDE:
    raise ValueError("Resume cần RUN_ID_OVERRIDE trỏ đúng run cũ trên Drive")
if RESUME_FROM_CHECKPOINT and MODE == "zero_shot":
    raise ValueError("Zero-shot không có checkpoint để resume")
if RESUME_FROM_CHECKPOINT and not RUN_FULL:
    raise ValueError("Resume cần RUN_FULL=True")
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ID = RUN_ID_OVERRIDE or f"{DATASET_ID}__{DIRECTION}__{MODE}__seed{TRAIN_SEED}__{MEMBER_TAG}__{EXPERIMENT_TAG}__{RUN_STAMP}"
OUTPUT_DIR = Path("/content/drive/MyDrive/NLP_runs") / DATASET_ID / DIRECTION / MODE / RUN_ID
required = [DATA_DIR / f"segments_dev_{DIRECTION}.jsonl"]
if MODE != "zero_shot":
    required.append(DATA_DIR / f"segments_train_{DIRECTION}.jsonl")
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Thiếu dữ liệu: " + ", ".join(missing))
print("Config:", CONFIG)
print("Run ID:", RUN_ID)
print("Output bền vững trên Drive:", OUTPUT_DIR)
training_preview = config_preview["training"]
print("T4 profile: train batch", training_preview.get("per_device_train_batch_size"),
      "x accumulation", training_preview.get("gradient_accumulation_steps"),
      "| eval batch", training_preview.get("per_device_eval_batch_size"),
      "| checkpoint eval samples", training_preview.get("checkpoint_eval_samples"))

## 6. Audit dữ liệu và chạy CLI có log thời gian

Audit kiểm tra checksum dataset, độ dài token source/target và chính sách đoạn quá dài trước khi tải toàn bộ model. Mỗi lệnh đều in thời điểm bắt đầu, kết thúc và tổng thời gian. Training dùng log gọn dạng `TRAIN epoch 0.52/3 | step ...` và cuối mỗi epoch in `EPOCH 1/3 completed | epoch time ... | elapsed ... | ETA ...`. Thời gian epoch bao gồm validation cuối epoch. Nếu lỗi, traceback gốc xuất hiện ngay phía trên dòng `THẤT BẠI`.

In [ ]:
from datetime import datetime
from time import perf_counter

def format_duration(seconds):
    minutes, seconds = divmod(round(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def run_cli(command, *extra):
    args = [sys.executable, "-u", "-m", "src.vinai_baseline.cli", command,
            "--config", str(CONFIG), "--output-dir", str(OUTPUT_DIR), *map(str, extra)]
    started = perf_counter()
    print(f"\n[{datetime.now().astimezone().isoformat(timespec='seconds')}] BẮT ĐẦU", flush=True)
    print("RUN:", " ".join(args), flush=True)
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    log_dir = Path("/content/vinai_cli_logs")
    log_dir.mkdir(parents=True, exist_ok=True)
    log_stamp = datetime.now().strftime("%Y%m%dT%H%M%S%f")
    log_path = log_dir / f"{command}_{log_stamp}.log"
    lines = []
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            args, cwd=PROJECT_DIR, env=environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()
            lines.append(line.rstrip())
        return_code = process.wait()
    if return_code != 0:
        elapsed = perf_counter() - started
        tail = "\n".join(lines[-80:]) or "(tiến trình không xuất log)"
        print(f"\nTHẤT BẠI sau {format_duration(elapsed)}", flush=True)
        print("Log đầy đủ:", log_path, flush=True)
        raise RuntimeError(
            f"Lệnh {command!r} thất bại với exit code {return_code}.\n"
            f"80 dòng log cuối:\n{tail}\n\nLog đầy đủ: {log_path}"
        ) from None
    elapsed = perf_counter() - started
    print(f"[{datetime.now().astimezone().isoformat(timespec='seconds')}] HOÀN THÀNH", flush=True)
    print("Tổng thời gian:", format_duration(elapsed), flush=True)
    return elapsed

# Preflight: chặn sớm trường hợp upload nhầm ZIP cũ chưa có bản sửa LoRA.
import importlib.metadata as package_metadata
collator_source = (PROJECT_DIR / "src/vinai_baseline/data.py").read_text(encoding="utf-8")
if "prepare_decoder_input_ids_from_labels" not in collator_source:
    raise RuntimeError("ZIP đang dùng là bản cũ, chưa có bản sửa labels/logits cho LoRA. Hãy upload NLP_colab.zip mới.")
print("Preflight LoRA collator: OK")
for package in ["transformers", "accelerate", "peft", "sacrebleu"]:
    print(f"{package}: {package_metadata.version(package)}")
print("torch:", torch.__version__, "| Python:", sys.version.split()[0])

# Audit tải tokenizer nhưng chưa tải toàn bộ model.
run_cli("audit")

## 7. Smoke test

Smoke test dùng 32 mẫu train và 16 mẫu validation (zero-shot chỉ dùng 16 validation). Mục tiêu là xác minh toàn bộ pipeline, save/reload model và không dùng metric smoke để kết luận chất lượng. Nếu muốn chạy lại smoke sau lỗi, hãy chạy lại cell cấu hình để tạo `RUN_ID` mới.

In [ ]:
# Smoke test nhỏ để phát hiện lỗi môi trường/config trước full run.
COMMAND = "evaluate" if MODE == "zero_shot" else "train"
if RESUME_FROM_CHECKPOINT:
    RESULT_DIR = OUTPUT_DIR
    print("Bỏ qua smoke test vì đang resume run đã được smoke trước đó.")
else:
    SMOKE_OUTPUT = Path("/content/vinai_smoke") / RUN_ID
    full_output = OUTPUT_DIR
    try:
        OUTPUT_DIR = SMOKE_OUTPUT
        if MODE == "zero_shot":
            SMOKE_ELAPSED = run_cli(COMMAND, "--max-validation-samples", "16")
        else:
            SMOKE_ELAPSED = run_cli(COMMAND, "--max-train-samples", "32", "--max-validation-samples", "16")
            run_cli("verify", "--max-validation-samples", "16")  # kiểm tra save/reload LoRA hoặc full model
        RESULT_DIR = SMOKE_OUTPUT
    finally:
        OUTPUT_DIR = full_output
    print("Smoke test hoàn thành tại:", RESULT_DIR)

## 8. Full run (chỉ chạy khi đã smoke thành công)

Cell chỉ train/evaluate toàn bộ khi `RUN_FULL=True`. Với LoRA trên T4, config giữ effective batch 16 bằng train batch 4 × accumulation 4, eval batch 16; dùng 128 mẫu cố định để chọn checkpoint mỗi epoch và chỉ đánh giá toàn bộ dev một lần ở cuối. Full run lưu trên Drive nên có thể resume sau khi Colab ngắt. Không dùng lại thư mục của run đã hoàn thành; hãy tạo `RUN_ID` mới cho thí nghiệm mới.

In [ ]:
# Full run chỉ bắt đầu khi bạn chủ động đặt RUN_FULL=True ở cell cấu hình.
if RUN_FULL:
    COMMAND = "evaluate" if MODE == "zero_shot" else "train"
    resume_args = (["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT]
                   if RESUME_FROM_CHECKPOINT else [])
    FULL_ELAPSED = run_cli(COMMAND, *resume_args)
    if MODE != "zero_shot":
        run_cli("verify")  # reload adapter/model đã lưu và dịch thử 1 mẫu
    RESULT_DIR = OUTPUT_DIR
else:
    print("Chưa chạy full. Cell kết quả sẽ hiển thị smoke test tại:", RESULT_DIR)

## 9. Xác minh kết quả

Cell đọc artifact của run gần nhất, kiểm tra đủ prediction, hiển thị BLEU/chrF++, chỉ số bảo toàn số và viện dẫn, runtime, VRAM, năm ví dụ dịch cố định theo seed 67 và kết quả theo category. Chất lượng chỉ nên được so sánh giữa các **full run dùng cùng dev set**, không so với smoke test.

In [ ]:
# Xác minh run gần nhất bằng metric và ví dụ dịch thực tế.
import json, random
import pandas as pd
from IPython.display import display

candidate_dirs = [path for path in [globals().get("RESULT_DIR"),
                                      globals().get("SMOKE_OUTPUT"), OUTPUT_DIR]
                  if path and (Path(path) / "reports/validation_metrics.json").exists()]
if not candidate_dirs:
    raise FileNotFoundError("Chưa có evaluation hoàn chỉnh. Hãy chạy smoke test hoặc full run trước.")
RESULT_DIR = max(candidate_dirs,
                 key=lambda path: (Path(path) / "reports/validation_metrics.json").stat().st_mtime)
RESULT_DIR = Path(RESULT_DIR)
metrics = json.loads((RESULT_DIR / "reports/validation_metrics.json").read_text(encoding="utf-8"))
manifest = json.loads((RESULT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
prediction_path = RESULT_DIR / "predictions/validation_predictions.jsonl"
predictions = [json.loads(line) for line in prediction_path.read_text(encoding="utf-8").splitlines() if line.strip()]

def metric_value(name):
    return metrics.get(f"validation_{name}", metrics.get(name))

summary = [
    ("BLEU", metric_value("bleu"), "Cao hơn tốt hơn; dùng so sánh các run cùng dev set"),
    ("chrF++", metric_value("chrf"), "Cao hơn tốt hơn; metric chính để chọn checkpoint"),
    ("Output rỗng (%)", metric_value("empty_output_rate"), "Lý tưởng là 0%"),
    ("Tỷ lệ độ dài", metric_value("length_ratio_mean"), "Gần 1 thường hợp lý; vẫn phải xem mẫu"),
    ("P95 tỷ lệ độ dài", metric_value("length_ratio_p95"), "Phát hiện đuôi output quá dài"),
    ("Output dài >3x (%)", metric_value("length_ratio_over_3_rate"), "Càng thấp càng tốt"),
    ("Lặp trigram (%)", metric_value("repeated_trigram_rate"), "Càng thấp càng tốt"),
    ("Token chiếm >20% (%)", metric_value("dominant_token_over_20_rate"), "Bắt output kiểu B B B..."),
    ("Bảo toàn số F1", metric_value("number_preservation_f1"), "Gần 100% tốt hơn"),
    ("Bảo toàn số từ source F1", metric_value("source_number_preservation_f1"), "Đo fidelity với đầu vào"),
    ("Bảo toàn viện dẫn F1", metric_value("reference_preservation_f1"), "Gần 100% tốt hơn"),
    ("Bảo toàn mã văn bản F1", metric_value("legal_identifier_preservation_f1"), "Ví dụ 63/2025/QH15"),
    ("Bảo toàn mã từ source F1", metric_value("source_legal_identifier_preservation_f1"), "Đo fidelity với đầu vào"),
    ("Thời gian evaluation (giây)", metric_value("runtime"), "Thời gian generate + tính metric"),
    ("Peak VRAM (MB)", metrics.get("peak_vram_mb"), "Bộ nhớ GPU cực đại khi evaluation"),
]
print("KẾT QUẢ:", RESULT_DIR)
print("Run:", manifest["run_name"], "| mode:", manifest["training_mode"],
      "| GPU:", manifest.get("gpu"))
display(pd.DataFrame(summary, columns=["Chỉ số", "Giá trị", "Cách đọc"]))

expected = int(metrics["evaluated_records"])
if len(predictions) != expected:
    raise RuntimeError(f"Thiếu prediction: có {len(predictions)}, kỳ vọng {expected}")
print(f"✓ Artifact đầy đủ: {len(predictions)}/{expected} prediction.")
print("Lưu ý: smoke test 16 mẫu chỉ xác minh pipeline. Muốn đánh giá chất lượng, chạy full dev set.")

sample_count = min(5, len(predictions))
sample_indices = random.Random(67).sample(range(len(predictions)), sample_count)
examples = pd.DataFrame([{
    "segment_id": predictions[index]["segment_id"],
    "source": predictions[index]["source"],
    "prediction": predictions[index]["prediction"],
    "reference": predictions[index]["target"],
} for index in sample_indices])
with pd.option_context("display.max_colwidth", 160):
    display(examples)

category_path = RESULT_DIR / "reports/validation_metrics_by_category.json"
if category_path.exists():
    by_category = json.loads(category_path.read_text(encoding="utf-8"))
    category_table = pd.DataFrame.from_dict(by_category, orient="index")
    display(category_table[[column for column in ["records", "bleu", "chrf",
        "number_preservation_f1", "reference_preservation_f1"]
        if column in category_table.columns]].sort_values("chrf"))